<a href="https://colab.research.google.com/github/abd500253-coder/Machine_Learning_Journey/blob/main/Titanic%20Survival%20Prediction%20Pipeline/Titanic_Survival_Prediction_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================

import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, Binarizer
from sklearn.naive_bayes import BernoulliNB

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


# =====================================================
# 2. LOAD DATASET
# =====================================================

df = sns.load_dataset("titanic")

# Feature engineering
df["with_family"] = np.where(
    (df["sibsp"] + df["parch"]) >= 1, 1, 0
)

# Remove unnecessary / redundant columns
df = df.drop(
    columns=[
        "deck", "alive", "embark_town", "sibsp",
        "parch", "alone", "sex", "adult_male",
        "class", "who"
    ]
)

print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())


# =====================================================
# 3. DEFINE FEATURES AND TARGET
# =====================================================

X = df.drop(columns=["survived"])
y = df["survived"]

numeric_features = ["age", "fare"]
categorical_features = ["embarked"]
binary_features = ["pclass", "with_family"]


# =====================================================
# 4. TRAIN / TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =====================================================
# 5. PREPROCESSING PIPELINES
# =====================================================

# Numerical: impute missing values, then scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical: impute, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Binary / ordinal values will be binarized
binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler()),
    ("binarizer", Binarizer())
])

# Combine all transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
        ("binary", binary_transformer, binary_features)
    ],
    remainder="drop"
)


# =====================================================
# 6. BUILD COMPLETE PIPELINE
# =====================================================

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", BernoulliNB())
])


# =====================================================
# 7. TRAIN BASELINE MODEL
# =====================================================

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n===== BASELINE RESULTS =====")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


# =====================================================
# 8. HYPERPARAMETER TUNING
# =====================================================

param_grid = {
    "classifier__alpha": [0.01, 0.1, 0.5, 1.0, 5.0],
    "classifier__fit_prior": [True, False],

    "preprocessor__numeric__imputer__strategy": [
        "mean", "median"
    ],

    "preprocessor__categorical__imputer__strategy": [
        "most_frequent", "constant"
    ]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_search.fit(X_train, y_train)


# =====================================================
# 9. BEST MODEL
# =====================================================

print("\n===== GRID SEARCH RESULTS =====")

print("Best CV Accuracy:", round(grid_search.best_score_, 3))
print("Best Parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_


# =====================================================
# 10. FINAL TEST EVALUATION
# =====================================================

best_pred = best_model.predict(X_test)

print("\n===== FINAL TEST RESULTS =====")
print("Test Accuracy:", round(accuracy_score(y_test, best_pred), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_pred))

print("\nClassification Report:")
print(classification_report(y_test, best_pred, zero_division=0))


# =====================================================
# 11. PREDICT CLASS PROBABILITIES
# =====================================================

probabilities = best_model.predict_proba(X_test)

probability_df = pd.DataFrame(
    probabilities,
    columns=[
        f"Class_{c}" for c in best_model.classes_
    ],
    index=X_test.index
)

print("\nFirst 5 predicted probabilities:")
print(probability_df.head())


# =====================================================
# 12. COMPARE ACTUAL AND PREDICTED RESULTS
# =====================================================

results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": best_pred
})

print("\nPrediction comparison:")
print(results.head(10))